In [0]:
# Notebook 1: Create Empty Bronze Tables
from delta.tables import DeltaTable
from pyspark.sql import SparkSession
import os

spark = SparkSession.builder.getOrCreate()

# Paths
raw_path = "abfss://raw@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/"
bronze_path = "abfss://processed@[YOUR_STORAGE_ACCOUNT_NAME].dfs.core.windows.net/bronze/"

try:
    all_items = dbutils.fs.ls(raw_path)
    print(f"Total items found in raw: {len(all_items)}")
    
    # Print names to see what the format actually is
    for item in all_items:
        print(f"Found: {item.name}")

    files = [f.path for f in all_items if f.name.lower().endswith(".csv")]
    print(f"Filtered CSV files: {len(files)}")

except Exception as e:
    print(f"Error accessing path: {e}")
    
# List all CSV files in raw
files = [f.path for f in dbutils.fs.ls(raw_path) if f.name.endswith(".csv")]

for file in files:
    # Create a table name based on filename
    table_name = os.path.basename(file).replace(".csv", "")

    # Read CSV with all string columns
    df = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", False)
          .option("sep", ",")
          .option("quote", '"')
          .option("escape", '"')
          .option("encoding", "UTF-8")
          .option("multiline", "true")
          .load(file)
          )
    
    # Check if Delta table already exists
    if not DeltaTable.isDeltaTable(spark, f"{bronze_path}{table_name}"):
        # Create empty Delta table
        df.write.format("delta").mode("overwrite").save(f"{bronze_path}{table_name}")
        print(f"--> Created empty table {table_name}")
    else:
        print(f"--> Table {table_name} already exists")

message = f"Successfully Created Empty Delta Tables"
dbutils.notebook.exit(message)